# 3 Topic Modeling and Clustering

In this section, we will do following steps to do clustering and give them labels:
1. **Embedding**: Get embeddings of the summary using SentenceTransformer;
2. **Clustering**: Perform dimension reduction clustering using HDBScan or KMeans;
3. **Labeling**: Label the clusters;

In [ ]:
import pandas as pd
import numpy as np

from itertools import product

import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer


In [2]:
summary_df = pd.read_parquet("021_filtered_data.parquet")
print("shape:", summary_df.shape)
print("columns:", summary_df.columns.tolist())

# 新数据用 summary 列做主题建模
text_col = "summary"
summary_df = summary_df.dropna(subset=[text_col]).copy()
summary_df[text_col] = summary_df[text_col].astype(str).str.strip()
summary_df = summary_df[summary_df[text_col].str.len() > 0].reset_index(drop=True)

word_len = summary_df[text_col].str.split().str.len()
print("avg words:", round(word_len.mean(), 2), "| p50:", int(word_len.quantile(0.5)), "| p90:", int(word_len.quantile(0.9)))


shape: (163071, 7)
columns: ['date', 'title', 'summary', 'organization', 'industry', 'impact', 'technology']
avg words: 45.74 | p50: 46 | p90: 56


In [3]:
summary_df.head()


,date,title,summary,organization,industry,impact,technology
0,2025-06-23,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",Bad Idea AI (BAD) is a cryptocurrency with a c...,Bad Idea AI,Cryptocurrency,0,AI
1,2024-07-01,This AI video of gymnastics might be the freak...,AI-generated videos of gymnastics and wrestlin...,Luma Dream Machine,Media and Entertainment,-2,AI-generated video
2,2024-09-22,"If using AI feels like a chore, try this - Boi...",1minAI offers a lifetime subscription for $39....,1minAI,Software,1,"GPT-4, Gemini Pro, Claude"
3,2023-11-10,The Road Ahead: How China's AI Foundation Mode...,"China's AI Foundation Model, developed by Baid...",Baidu,Automotive,2,"deep learning, big data, cloud computing"
4,2023-11-19,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia are collaborating to empo...,Microsoft and Nvidia,Technology,1,Windows AI Studio and TensorRT-LLM


In [4]:
TEST_N = 10000
RANDOM_SEED = 42

if len(summary_df) > TEST_N:
    test_df = summary_df.sample(n=TEST_N, random_state=RANDOM_SEED).reset_index(drop=True)
else:
    test_df = summary_df.copy()

docs_test = test_df[text_col].tolist()
print("test docs:", len(docs_test))


test docs: 10000


In [5]:
# if torch.cuda.is_available():
#     device = "cuda"
# elif torch.backends.mps.is_available():
#     device = "mps"
# else:
#     device = "cpu"

# # 短文本依然建议用 sentence-transformers，语义聚类会比纯词袋稳定
# model_name = "sentence-transformers/all-MiniLM-L6-v2"
# embedder = SentenceTransformer(model_name, device=device)

# emb_test = embedder.encode(
#     docs_test,
#     batch_size=256 if device in ["cuda", "mps"] else 128,
#     show_progress_bar=True,
#     convert_to_numpy=True,
#     normalize_embeddings=True
# )

# print("device:", device)
# print("embedding model:", model_name)
# print("embedding shape:", emb_test.shape)


In [6]:
import torch
from sentence_transformers import SentenceTransformer

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

model_name = "Qwen/Qwen3-Embedding-0.6B"
embedder = SentenceTransformer(
    model_name,
    device=device,
    trust_remote_code=True,
)

emb_test = embedder.encode(
    docs_test,
    batch_size=64 if device in ["cuda", "mps"] else 32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("device:", device)
print("embedding model:", model_name)
print("embedding shape:", emb_test.shape)


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

device: mps
embedding model: Qwen/Qwen3-Embedding-0.6B
embedding shape: (10000, 1024)


In [ ]:
# TARGET_OUTLIER = 0.2

# def eval_run(topics, emb):
#     labels = np.array(topics)
#     outlier_rate = np.mean(labels == -1)
#     valid = labels != -1
#     n_topics = len(set(labels[valid])) if valid.any() else 0

#     sil = np.nan
#     if valid.sum() > 20 and n_topics > 1:
#         sil = silhouette_score(emb[valid], labels[valid])

#     outlier_penalty = abs(outlier_rate - TARGET_OUTLIER)
#     combined_score = (sil * 0.7) - (outlier_penalty * 0.3) if not np.isnan(sil) else -outlier_penalty

#     return {
#         "n_topics": n_topics,
#         "outlier_rate": float(outlier_rate),
#         "silhouette": float(sil) if not np.isnan(sil) else np.nan,
#         "combined_score": float(combined_score) if not np.isnan(combined_score) else np.nan,
#     }

# param_grid = {
#     "n_neighbors": [30,40,50],
#     "min_dist": [0.0],
#     "min_cluster_size": [20,30,40],
#     "min_samples": [1,2,3,5],
# }

# results = []
# vectorizer_model = CountVectorizer(
#     stop_words="english",
#     ngram_range=(1, 2),
#     min_df=5,
#     max_df=0.90,
#     token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
# )

# all_settings = list(product(
#     param_grid["n_neighbors"],
#     param_grid["min_dist"],
#     param_grid["min_cluster_size"],
#     param_grid["min_samples"],
# ))

# total_runs = len(all_settings)
# for run_id, (n_neighbors, min_dist, min_cluster_size, min_samples) in enumerate(all_settings, start=1):
#     print(f"\n[{run_id}/{total_runs}] start -> nn={n_neighbors}, md={min_dist}, mcs={min_cluster_size}, ms={min_samples}")

#     umap_model = UMAP(
#         n_neighbors=n_neighbors,
#         n_components=5,
#         min_dist=min_dist,
#         metric="cosine",
#         random_state=42,
#         low_memory=True,
#     )
#     hdbscan_model = HDBSCAN(
#         min_cluster_size=min_cluster_size,
#         min_samples=min_samples,
#         metric="euclidean",
#         cluster_selection_method="eom",
#         prediction_data=False,
#     )

#     tm = BERTopic(
#         umap_model=umap_model,
#         hdbscan_model=hdbscan_model,
#         vectorizer_model=vectorizer_model,
#         calculate_probabilities=False,
#         verbose=False,
#     )

#     try:
#         topics, _ = tm.fit_transform(docs_test, emb_test)
#         m = eval_run(topics, emb_test)
#         m.update({
#             "n_neighbors": n_neighbors,
#             "min_dist": min_dist,
#             "min_cluster_size": min_cluster_size,
#             "min_samples": min_samples,
#         })
#         results.append(m)
#         print(f"[{run_id}/{total_runs}] done  -> topics={m['n_topics']}, outlier={m['outlier_rate']:.4f}, sil={m['silhouette']:.4f}")
#     except Exception as e:
#         print(f"[{run_id}/{total_runs}] skip  -> {type(e).__name__}: {e}")

# res_df = pd.DataFrame(results)
# res_df["topic_range_score"] = res_df["n_topics"].apply(
#     lambda x: 1 if 30 <= x <= 90 else (0.5 if 20 <= x < 30 or 90 < x <= 120 else 0)
# )

# res_df = res_df.sort_values(
#     by=["combined_score", "topic_range_score", "silhouette"],
#     ascending=[False, False, False],
# ).reset_index(drop=True)



[1/36] start -> nn=30, md=0.0, mcs=20, ms=1
[1/36] done  -> topics=136, outlier=0.3117, sil=0.0489

[2/36] start -> nn=30, md=0.0, mcs=20, ms=2
[2/36] done  -> topics=134, outlier=0.3324, sil=0.0507

[3/36] start -> nn=30, md=0.0, mcs=20, ms=3
[3/36] done  -> topics=115, outlier=0.3568, sil=0.0588

[4/36] start -> nn=30, md=0.0, mcs=20, ms=5
[4/36] done  -> topics=110, outlier=0.3685, sil=0.0588

[5/36] start -> nn=30, md=0.0, mcs=30, ms=1
[5/36] done  -> topics=89, outlier=0.3289, sil=0.0538

[6/36] start -> nn=30, md=0.0, mcs=30, ms=2
[6/36] done  -> topics=79, outlier=0.3518, sil=0.0549

[7/36] start -> nn=30, md=0.0, mcs=30, ms=3
[7/36] done  -> topics=73, outlier=0.3445, sil=0.0529

[8/36] start -> nn=30, md=0.0, mcs=30, ms=5
[8/36] done  -> topics=71, outlier=0.3684, sil=0.0579

[9/36] start -> nn=30, md=0.0, mcs=40, ms=1
[9/36] done  -> topics=64, outlier=0.3385, sil=0.0548

[10/36] start -> nn=30, md=0.0, mcs=40, ms=2
[10/36] done  -> topics=56, outlier=0.3240, sil=0.0499

[11

In [11]:
TARGET_OUTLIER = 0.2

def eval_run(topics, emb):
    labels = np.array(topics)
    outlier_rate = np.mean(labels == -1)
    valid = labels != -1
    n_topics = len(set(labels[valid])) if valid.any() else 0

    sil = np.nan
    if valid.sum() > 20 and n_topics > 1:
        sil = silhouette_score(emb[valid], labels[valid])

    outlier_penalty = abs(outlier_rate - TARGET_OUTLIER)
    combined_score = (sil * 0.7) - (outlier_penalty * 0.3) if not np.isnan(sil) else -outlier_penalty

    return {
        "n_topics": n_topics,
        "outlier_rate": float(outlier_rate),
        "silhouette": float(sil) if not np.isnan(sil) else np.nan,
        "combined_score": float(combined_score) if not np.isnan(combined_score) else np.nan,
    }

param_grid = {
    "n_neighbors": [40, 60],
    "min_dist": [0.0],
    "n_components": [5],
    "min_cluster_size": [10, 20],
    "min_samples": [1, 2],
    "cluster_selection_epsilon": [0.0, 0.03],
}


results = []
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
)

all_settings = list(product(
    param_grid["n_neighbors"],
    param_grid["min_dist"],
    param_grid["n_components"],
    param_grid["min_cluster_size"],
    param_grid["min_samples"],
    param_grid["cluster_selection_epsilon"],
))

total_runs = len(all_settings)
for run_id, (n_neighbors, min_dist, n_components, min_cluster_size, min_samples, cluster_selection_epsilon) in enumerate(all_settings, start=1):
    print(f"\n[{run_id}/{total_runs}] start -> nn={n_neighbors}, md={min_dist}, nc={n_components}, mcs={min_cluster_size}, ms={min_samples}, cse={cluster_selection_epsilon}")

    umap_model = UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        min_dist=min_dist,
        metric="cosine",
        random_state=42,
        low_memory=True,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=False,
    )
    umap_model = UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        min_dist=min_dist,
        metric="cosine",
        random_state=42,
        low_memory=True,
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom",
        cluster_selection_epsilon=cluster_selection_epsilon,
        prediction_data=False,
    )


    tm = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=False,
    )

    try:
        topics, _ = tm.fit_transform(docs_test, emb_test)
        m = eval_run(topics, emb_test)
        m.update({
            "n_neighbors": n_neighbors,
            "min_dist": min_dist,
            "n_components": n_components,
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
            "cluster_selection_epsilon": cluster_selection_epsilon,
        })
        results.append(m)
        print(f"[{run_id}/{total_runs}] done  -> topics={m['n_topics']}, outlier={m['outlier_rate']:.4f}, sil={m['silhouette']:.4f}")
    except Exception as e:
        print(f"[{run_id}/{total_runs}] skip  -> {type(e).__name__}: {e}")

res_df = pd.DataFrame(results)
res_df["topic_range_score"] = res_df["n_topics"].apply(
    lambda x: 1 if 30 <= x <= 90 else (0.5 if 20 <= x < 30 or 90 < x <= 120 else 0)
)

res_df = res_df.sort_values(
    by=["combined_score", "topic_range_score", "silhouette"],
    ascending=[False, False, False],
).reset_index(drop=True)


[1/16] start -> nn=40, md=0.0, nc=5, mcs=10, ms=1, cse=0.0
[1/16] done  -> topics=240, outlier=0.3059, sil=0.0492

[2/16] start -> nn=40, md=0.0, nc=5, mcs=10, ms=1, cse=0.03
[2/16] done  -> topics=237, outlier=0.3045, sil=0.0518

[3/16] start -> nn=40, md=0.0, nc=5, mcs=10, ms=2, cse=0.0
[3/16] done  -> topics=222, outlier=0.3351, sil=0.0525

[4/16] start -> nn=40, md=0.0, nc=5, mcs=10, ms=2, cse=0.03
[4/16] done  -> topics=219, outlier=0.3337, sil=0.0552

[5/16] start -> nn=40, md=0.0, nc=5, mcs=20, ms=1, cse=0.0
[5/16] done  -> topics=112, outlier=0.2757, sil=0.0443

[6/16] start -> nn=40, md=0.0, nc=5, mcs=20, ms=1, cse=0.03
[6/16] done  -> topics=112, outlier=0.2757, sil=0.0443

[7/16] start -> nn=40, md=0.0, nc=5, mcs=20, ms=2, cse=0.0
[7/16] done  -> topics=106, outlier=0.3011, sil=0.0469

[8/16] start -> nn=40, md=0.0, nc=5, mcs=20, ms=2, cse=0.03
[8/16] done  -> topics=106, outlier=0.3011, sil=0.0469

[9/16] start -> nn=60, md=0.0, nc=5, mcs=10, ms=1, cse=0.0
[9/16] done  -> 

In [12]:
res_df.head(20)


,n_topics,outlier_rate,silhouette,combined_score,n_neighbors,min_dist,n_components,min_cluster_size,min_samples,cluster_selection_epsilon,topic_range_score
0,112,0.2757,0.044274,0.008282,40,0.0,5,20,1,0.00,0.5
1,112,0.2757,0.044274,0.008282,40,0.0,5,20,1,0.03,0.5
2,237,0.3045,0.051757,0.004880,40,0.0,5,10,1,0.03,0.0
3,240,0.3059,0.049204,0.002673,40,0.0,5,10,1,0.00,0.0
4,106,0.3011,0.046940,0.002528,40,0.0,5,20,2,0.00,0.5
5,106,0.3011,0.046940,0.002528,40,0.0,5,20,2,0.03,0.5
6,123,0.3383,0.058832,-0.000307,60,0.0,5,20,1,0.00,0.0
7,123,0.3383,0.058832,-0.000307,60,0.0,5,20,1,0.03,0.0
8,231,0.3212,0.051017,-0.000648,60,0.0,5,10,1,0.00,0.0
9,231,0.3212,0.051017,-0.000648,60,0.0,5,10,1,0.03,0.0


In [13]:
best = res_df.iloc[0]
print("Best params:")
display(best)

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
)
ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)

umap_model = UMAP(
    n_neighbors=int(best["n_neighbors"]),
    n_components=5,
    min_dist=float(best["min_dist"]),
    metric="cosine",
    random_state=42,
    low_memory=True,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=int(best["min_cluster_size"]),
    min_samples=int(best["min_samples"]),
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=False,
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(docs_test, emb_test)
topic_info = topic_model.get_topic_info()
display(topic_info.head(20))


Best params:


n_topics                     112.000000
outlier_rate                   0.275700
silhouette                     0.044274
combined_score                 0.008282
n_neighbors                   40.000000
min_dist                       0.000000
n_components                   5.000000
min_cluster_size              20.000000
min_samples                    1.000000
cluster_selection_epsilon      0.000000
topic_range_score              0.500000
Name: 0, dtype: float64

2026-03-11 01:06:49,519 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 01:07:07,467 - BERTopic - Dimensionality - Completed ✓
2026-03-11 01:07:07,469 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 01:07:07,610 - BERTopic - Cluster - Completed ✓
2026-03-11 01:07:07,613 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-11 01:07:07,928 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2757,-1_news_energy_revenue_service,"[news, energy, revenue, service, advertising, ...",[OpenAI has announced significant partnerships...
1,0,466,0_cancer_patient_researchers developed_predict,"[cancer, patient, researchers developed, predi...","[LG AI Research has launched Exaone Path 2.0, ..."
2,1,426,1_download_describes_formats_available free,"[download, describes, formats, available free,...",[This content describes an AI-generated image ...
3,2,272,2_survey_jobs_job_reveals,"[survey, jobs, job, reveals, workers, indicate...",[A recent study found that 36% of IT workers f...
4,3,234,3_agent_solution_agents_platform designed,"[agent, solution, agents, platform designed, s...","[RUGR Panorama AI, a new on-premise Agentic AI..."
5,4,216,4_order_biden_executive_white house,"[order, biden, executive, white house, house, ...","[Major tech companies including Amazon, Google..."
6,5,190,5_projected_billion driven_projected reach_grow,"[projected, billion driven, projected reach, g...",[The Machine Learning (ML) Intelligent Process...
7,6,171,6_edge ai_processors_intel_edge,"[edge ai, processors, intel, edge, gpus, pcs, ...","[Intel has launched new AI chips, including Ga..."
8,7,144,7_meta_llama_metas_whatsapp,"[meta, llama, metas, whatsapp, glasses, metave...","[Facebook's parent company, Meta, has develope..."
9,8,143,8_authors_copyrighted_lawsuit_copyright,"[authors, copyrighted, lawsuit, copyright, inf...",[The New York Times has sued OpenAI and Micros...


In [ ]:
# # 1) 先正常建模
# topics, _ = topic_model.fit_transform(docs_test, emb_test)

# # 2) 统计原始 outlier rate
# orig_outlier = (np.array(topics) == -1).mean()
# print(f"Original outlier rate: {orig_outlier:.4f}")

# # 3) 二次分配离群点（推荐先用 embeddings）
# new_topics = topic_model.reduce_outliers(
#     docs_test,
#     topics,
#     embeddings=emb_test,
#     strategy="embeddings",
# )


# # 4) 用新topic更新模型
# topic_model.update_topics(docs_test, topics=new_topics)

# # 5) 再看更新后的结果
# new_outlier = (np.array(new_topics) == -1).mean()
# print(f"New outlier rate: {new_outlier:.4f}")

# topic_info = topic_model.get_topic_info()
# display(topic_info)


2026-03-09 04:07:19,402 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-09 04:07:32,351 - BERTopic - Dimensionality - Completed ✓
2026-03-09 04:07:32,352 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-09 04:07:32,468 - BERTopic - Cluster - Completed ✓
2026-03-09 04:07:32,469 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-09 04:07:32,618 - BERTopic - Representation - Completed ✓
2026-03-09 04:07:32,772 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Original outlier rate: 0.2942
New outlier rate: 0.0000


,Topic,Count,Name,Representation,Representative_Docs
0,0,474,0_image_aigenerated_rawpixelcom_available,"[image, aigenerated, rawpixelcom, available, d...",[This content is a description of an AI-genera...
1,1,456,1_patient_healthcare_medical_health,"[patient, healthcare, medical, health, care, d...",[OneMedNet has partnered with Palantir to leve...
2,2,215,2_trading_cryptocurrency_blockchain_market,"[trading, cryptocurrency, blockchain, market, ...",[The article details the availability and util...
3,3,250,3_regulations_regulation_act_eu,"[regulations, regulation, act, eu, bill, ai, l...","[The EU has agreed on world-leading AI rules, ..."
4,4,220,4_openai_chatgpt_openais_model,"[openai, chatgpt, openais, model, users, gpt4,...","[OpenAI has launched ChatGPT Enterprise, a new..."
...,...,...,...,...,...
99,99,43,99_her_was_aigenerated_illustrated,"[her, was, aigenerated, illustrated, sports, m...",[Sports Illustrated has faced backlash for all...
100,100,90,100_smart_home_ces_aipowered,"[smart, home, ces, aipowered, launched, has, i...",[Landing AI has released an SDK and code sampl...
101,101,48,101_edge_iot_applications_network,"[edge, iot, applications, network, wifi, proce...",[Cognizant's Neuro® Edge platform simplifies a...
102,102,23,102_us_china_restrictions_export,"[us, china, restrictions, export, chips, chip,...",[A US lawmaker is urging the implementation of...


In [ ]:
doc_info = topic_model.get_document_info(docs_test)
# display(doc_info[["Document", "Topic", "Name", "Probability"]].head(20))

# # 看一个非离群主题的样本文档
# top_topic = topic_info.loc[topic_info["Topic"] != -1, "Topic"].iloc[0]
# sample_docs = doc_info[doc_info["Topic"] == top_topic]["Document"].head(10).tolist()

# for i, d in enumerate(sample_docs, 1):
#     print(f"\n--- doc {i} ---\n{d[:500]}")

t = 50
sample_docs = doc_info[doc_info["Topic"] == t]["Document"].tolist()
for i, d in enumerate(sample_docs, 1):
    print(f"\n--- doc {i} ---\n{d}" + "\n")


--- doc 1 ---
Bryant University is launching four new graduate programs in Business Analytics, Data Science, Healthcare Informatics, and Taxation, with three STEM-designated Master's degrees starting in Fall 2023. These programs aim to equip students and professionals with skills for the data-driven digital economy and align with the university's Vision 2030 Strategic Plan.


--- doc 2 ---
This project-based Guided Project on Coursera teaches you to implement ensemble methods like Bagging, Boosting, and Stacking in machine learning. It's a 2-hour intermediate course with split-screen video instruction and a browser-based cloud desktop for hands-on practice.


--- doc 3 ---
This content is an advertisement for "25 Machine Learning Projects For All Levels Datacamp," listing its price, customer reviews, and related products/courses. It also includes irrelevant information about shoe and clothing sizes, and a list of unrelated chart types.


--- doc 4 ---
This IBM course focuses on deploy

In [19]:
top_n = min(10, topic_info[topic_info["Topic"] != -1].shape[0])
fig = topic_model.visualize_barchart(top_n_topics=top_n)
fig.show()


In [20]:
fig = topic_model.visualize_topics()
fig.show()


In [21]:
doc_info[doc_info["Topic"] == 1]

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
26,This content describes an AI-generated image r...,1,1_download_describes_formats_available free,"[download, describes, formats, available free,...",[This content describes an AI-generated image ...,download - describes - formats - available fre...,1.0,False
110,"This is a free, AI-generated image from rawpix...",1,1_download_describes_formats_available free,"[download, describes, formats, available free,...",[This content describes an AI-generated image ...,download - describes - formats - available fre...,1.0,False
129,This is a description of an AI-generated image...,1,1_download_describes_formats_available free,"[download, describes, formats, available free,...",[This content describes an AI-generated image ...,download - describes - formats - available fre...,1.0,False
141,This is a description of an AI-generated image...,1,1_download_describes_formats_available free,"[download, describes, formats, available free,...",[This content describes an AI-generated image ...,download - describes - formats - available fre...,1.0,False
152,This content describes an AI-generated image f...,1,1_download_describes_formats_available free,"[download, describes, formats, available free,...",[This content describes an AI-generated image ...,download - describes - formats - available fre...,1.0,False
...,...,...,...,...,...,...,...,...
9799,This content describes an AI-generated image o...,1,1_download_describes_formats_available free,"[download, describes, formats, available free,...",[This content describes an AI-generated image ...,download - describes - formats - available fre...,1.0,False
9883,This is a description of an AI-generated image...,1,1_download_describes_formats_available free,"[download, describes, formats, available free,...",[This content describes an AI-generated image ...,download - describes - formats - available fre...,1.0,False
9899,This content describes an AI-generated image o...,1,1_download_describes_formats_available free,"[download, describes, formats, available free,...",[This content describes an AI-generated image ...,download - describes - formats - available fre...,1.0,False
9903,"This is a free, AI-generated royalty-free phot...",1,1_download_describes_formats_available free,"[download, describes, formats, available free,...",[This content describes an AI-generated image ...,download - describes - formats - available fre...,1.0,False


In [23]:
from transformers import pipeline
import pandas as pd

df = doc_info.loc[:, ["Document", "Topic"]].copy()

clf = pipeline(
    "text-classification",
    model="sentiment_distilroberta_financial_phrasebank",
    tokenizer="sentiment_distilroberta_financial_phrasebank",
)

preds = clf(df["Document"].tolist(), batch_size=32, truncation=True, max_length=128)

df["sentiment"] = [x["label"] for x in preds]
df["score"] = [x["score"] for x in preds]

cluster_sent = (
    df.groupby(["Topic", "sentiment"])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=["negative", "neutral", "positive"], fill_value=0)
)

cluster_sent


Device set to use mps:0


sentiment,negative,neutral,positive
Topic,,,
-1,435,1113,1209
0,7,117,342
1,0,426,0
2,87,72,113
3,0,64,170
...,...,...,...
107,9,9,2
108,0,1,19
109,3,15,2


In [33]:
# for doc in df[df["Topic"] == 111]['Document']:
#     print(doc + "\n")


for _, row in df[df["Topic"] == 107][["Document", "sentiment", "score"]].head(20).iterrows():
    print(f"[{row['sentiment']}] score={row['score']:.4f}")
    print(row["Document"])
    print("-" * 80)


[neutral] score=0.9809
Scammers are now using AI to clone the voices of loved ones for fake calls claiming a relative is in trouble and needs money urgently. To verify, contact the alleged victim or another family member directly and be wary of payment requests via wire transfer or gift cards.
--------------------------------------------------------------------------------
[positive] score=0.7586
The FCC has officially named the "Royal Tiger" gang as the first AI robocall scammer, utilizing sophisticated AI voice cloning and other tactics to defraud millions. While this naming aims to raise awareness and encourage international action, experts warn of a surge in AI-powered scams, urging individuals to remain vigilant and protect their personal information.
--------------------------------------------------------------------------------
[neutral] score=0.7654
The Better Business Bureau (BBB) is warning consumers about AI-generated voice scams that can impersonate loved ones in distress 

# K-mean part

In [34]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


In [35]:
TARGET_TOPICS_LOW = 30
TARGET_TOPICS_HIGH = 90

def eval_run_kmeans(topics, emb):
    labels = np.array(topics)
    n_topics = len(np.unique(labels))

    sil = np.nan
    if len(labels) > 20 and n_topics > 1:
        sil = silhouette_score(emb, labels)

    topic_range_score = 1 if TARGET_TOPICS_LOW <= n_topics <= TARGET_TOPICS_HIGH else (
        0.5 if 20 <= n_topics < TARGET_TOPICS_LOW or TARGET_TOPICS_HIGH < n_topics <= 120 else 0
    )

    return {
        "n_topics": int(n_topics),
        "outlier_rate": 0.0,
        "silhouette": float(sil) if not np.isnan(sil) else np.nan,
        "topic_range_score": float(topic_range_score),
        "combined_score": float(sil + 0.15 * topic_range_score) if not np.isnan(sil) else np.nan,
    }

param_grid = {
    "use_scaler": [False, True],
    "n_components": [20, 30, 50],
    "n_clusters": [40, 60, 80],
}

results = []

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
)

all_settings = list(product(
    param_grid["use_scaler"],
    param_grid["n_components"],
    param_grid["n_clusters"],
))

total_runs = len(all_settings)

for run_id, (use_scaler, n_components, n_clusters) in enumerate(all_settings, start=1):
    print(f"\n[{run_id}/{total_runs}] start -> scaler={use_scaler}, pca={n_components}, k={n_clusters}")

    steps = []
    if use_scaler:
        steps.append(("scaler", StandardScaler()))
    steps.append(("pca", PCA(n_components=n_components, random_state=42)))
    dim_model = Pipeline(steps)

    cluster_model = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init="auto",
    )

    tm = BERTopic(
        umap_model=dim_model,        # 这里虽然叫 umap_model，但可放任意降维器
        hdbscan_model=cluster_model, # 这里虽然叫 hdbscan_model，但可放任意聚类器
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=False,
    )

    try:
        topics, _ = tm.fit_transform(docs_test, emb_test)
        m = eval_run_kmeans(topics, emb_test)
        m.update({
            "use_scaler": use_scaler,
            "n_components": n_components,
            "n_clusters": n_clusters,
        })
        results.append(m)
        print(f"[{run_id}/{total_runs}] done -> topics={m['n_topics']}, sil={m['silhouette']:.4f}")
    except Exception as e:
        print(f"[{run_id}/{total_runs}] skip -> {type(e).__name__}: {e}")

res_df = pd.DataFrame(results).sort_values(
    by=["combined_score", "topic_range_score", "silhouette"],
    ascending=[False, False, False],
).reset_index(drop=True)

res_df.head(20)



[1/18] start -> scaler=False, pca=20, k=40


: 

In [ ]:
best = res_df.iloc[0]
print("Best params:")
display(best)

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
)

ctfidf_model = ClassTfidfTransformer(
    bm25_weighting=True,
    reduce_frequent_words=True,
)

steps = []
if bool(best["use_scaler"]):
    steps.append(("scaler", StandardScaler()))
steps.append(("pca", PCA(n_components=int(best["n_components"]), random_state=42)))
dim_model = Pipeline(steps)

cluster_model = KMeans(
    n_clusters=int(best["n_clusters"]),
    random_state=42,
    n_init="auto",
)

topic_model = BERTopic(
    umap_model=dim_model,
    hdbscan_model=cluster_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(docs_test, emb_test)
topic_info = topic_model.get_topic_info()
display(topic_info.head(20))
